# Agentic Multi-Hop RAG — Colab Experiment
**Before running:** `Runtime → Change runtime type → A100 GPU`

In [ ]:
# Cell 1: Mount Drive and set project path
from google.colab import drive
drive.mount('/content/drive')

import os

PROJECT_PATH = '/content/drive/MyDrive/CS572/Final'
os.chdir(PROJECT_PATH)

print('Working directory:', os.getcwd())
print('Files:', os.listdir('.'))


In [ ]:
# Cell 2: Install dependencies
!pip install -q -U transformers bitsandbytes accelerate
!pip install -q -U rank-bm25 datasets sentence-transformers python-dotenv google-genai
print('Done.')


In [ ]:
# Cell 3: Load .env and log in to Hugging Face
import os
from dotenv import load_dotenv
from huggingface_hub import login

load_dotenv(os.path.join(PROJECT_PATH, '.env'))

hf_token = os.getenv('HF_TOKEN')
gemini_api_key = os.getenv('GEMINI_API_KEY')

if not hf_token:
    raise ValueError('Missing HF_TOKEN in .env')
# Gemini is optional. Only needed when RUN_FRONTIER = True below.

login(token=hf_token)

os.environ['HF_TOKEN'] = hf_token
os.environ['HUGGING_FACE_HUB_TOKEN'] = hf_token
if gemini_api_key:
    os.environ['GEMINI_API_KEY'] = gemini_api_key

print('HF_TOKEN loaded:', bool(hf_token))
print('GEMINI_API_KEY loaded:', bool(gemini_api_key))


In [ ]:
# Cell 4: Verify GPU
import torch

print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None')
print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB' if torch.cuda.is_available() else '')


In [ ]:
# Cell 5: Add project root to Python path and clear stale caches
import sys
import shutil

if PROJECT_PATH not in sys.path:
    sys.path.insert(0, PROJECT_PATH)

for root, dirs, _ in os.walk(PROJECT_PATH):
    for d in dirs:
        if d == '__pycache__':
            shutil.rmtree(os.path.join(root, d))

for mod in list(sys.modules.keys()):
    if mod.startswith('src'):
        del sys.modules[mod]

print('Path set and cache cleared.')


In [ ]:
# Cell 6: Pilot run. Set RUN_FRONTIER=True only if Gemini has available credits.
from src.run_experiment import run

RUN_FRONTIER = True

pilot_results = run(
    num_samples=5,
    baseline_provider="hf",
    baseline_model_name="meta-llama/Meta-Llama-3.1-8B-Instruct",
    agent_provider="hf",
    agent_model_name="meta-llama/Meta-Llama-3.1-8B-Instruct",
    frontier_provider="gemini" if RUN_FRONTIER else None,
    frontier_model_name="gemini-2.5-pro",
    frontier_num_samples=5,
)

print('Pilot results:', pilot_results)


In [ ]:
# Cell 7: Full run with Gemini frontier ceiling.
from src.run_experiment import run

RUN_FRONTIER = True

results = run(
    num_samples=50,
    baseline_provider="hf",
    baseline_model_name="meta-llama/Meta-Llama-3.1-8B-Instruct",
    agent_provider="hf",
    agent_model_name="meta-llama/Meta-Llama-3.1-8B-Instruct",
    frontier_provider="gemini" if RUN_FRONTIER else None,
    frontier_model_name="gemini-2.5-pro",
    frontier_num_samples=20,  # adjust higher if budget/time allows
)

print('Final results:', results)


In [ ]:
# Cell 8: Inspect saved aggregate results
import json

results_path = os.path.join(PROJECT_PATH, 'results', 'experiment_results.json')
with open(results_path) as f:
    saved = json.load(f)

print(json.dumps(saved, indent=2))
print('Saved to:', results_path)


In [ ]:
# Cell 9: White-box breakdowns and case studies
from src.reporting import (
    load_detailed_results,
    print_support_doc_comparison,
    print_support_doc_breakdown,
    print_case_studies,
    print_intermediate_trace,
)

details = load_detailed_results()

print_support_doc_comparison(details)
print_support_doc_breakdown(details, system_name="baseline")
print_support_doc_breakdown(details, system_name="agentic")

if "frontier_agentic" in details:
    print_support_doc_breakdown(details, system_name="frontier_agentic")

print_case_studies(details, system_name="agentic", group_name="successful_multi_doc")
print_case_studies(details, system_name="agentic", group_name="failed_multi_doc")
print_case_studies(details, system_name="agentic", group_name="successful_single_doc")
print_case_studies(details, system_name="agentic", group_name="failed_single_doc")
print_intermediate_trace(details, system_name="agentic", trace_index=0)


In [ ]:
# Cell 10: Optional raw trace inspection for one sample
details = load_detailed_results()
trace = details["agentic"]["traces"][0]

print("Question:", trace["question"])
print("Gold:", trace["gold_answer"])
print("Pred:", trace["predicted_answer"])
print("Sub-queries:", trace["sub_queries"])
print("Requires multiple docs:", trace["requires_multiple_documents"])
print("Num hops:", trace["num_hops"])

for hop in trace["per_hop"]:
    print("\nHop query:", hop["query"])
    for doc in hop["docs"]:
        print("-", doc["title"], "|", doc["text_preview"])
